<a href="https://colab.research.google.com/github/us788/AI-NLP-likelion-bootcamp/blob/main/Subword_problem_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 서브 워드 비교 실험

서브워드 라이브러리 HuggingFace Tokenizer와 SentencePiece를 비교하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 준비 : 네이버 영화 리뷰 [[링크]](https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt)
2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정
3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석
4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정
5. 기타 추가 실험
6. 사용성과 장단점 분석

## 1. 데이터 준비 : 네이버 영화 리뷰

### HuggingFace Tokenizer와 SentencePiece 설치하기

In [ ]:
!pip install -q sentencepiece tokenizers

### 네이버 리뷰 데이터 불러오기

In [ ]:
import requests
import time

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
response = requests.get(url)

with open("corpus.txt", "wb") as f:
    f.write(response.content)

In [ ]:
import os

file_path = "/content/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


### 데이터 정제하기
현재 데이터에는 `id`, `document`, `label`이 있습니다. 이 중 실질적인 문장 데이터인 `document`만 활용합니다.

In [ ]:
# [[YOUR CODE]]
import pandas as pd

df = pd.read_csv("corpus.txt", sep="\t")

df = df.drop(columns=['id'])
df = df.drop(columns=['label'])

X = df['document']

## 2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정

HuggingFace BPE는 SentencePiece와 내제된 철학이 다릅니다.  
고전적인 BPE는 단어 단위 알고리즘에서 출발하였으며 단어 목록을 입력으로 받아 문장 분리 -> 단어 분리 -> BPE를 가정합니다.  
때문에 이 철학을 따르는 HuggingFace BPE는 `Witespace()` 혹은 `ByteLevel()`과 같은 pre-tokenizer가 필요합니다.  
반면 SentencePiece는 공백도 학습하자는 철학으로 공백에 대하여 \_\_로 처리하여 기본 문장에서 공백을 기준으로 나누는 것은 동일하지만 문장 전체를 입력받게 되며 다음과 같이 처리합니다.  

"오늘은 날이 참 밝다."  
"오늘은\_\_날이\_\_참\_\_밝다."

만일 SentencePiece의 철학과 유사하게 진행하려면 `Metqaspace()`를 활용하십시오.

### Sentencepiece를 사용하여 단어사전 구축

In [ ]:
# [[YOUR CODE]]

from tokenizers.pre_tokenizers import Metaspace
import re

def sentencepiece_tokenizer(text):
    if pd.isnull(text):
        return []

    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)

    tokens = Metaspace().pre_tokenize_str(str(text))

    return [token for token, _ in tokens]

In [ ]:
s_start = time.perf_counter()
tokenized_docs = X.apply(sentencepiece_tokenizer)
s_end = time.perf_counter()

In [ ]:
all_tokens = []

for doc in tokenized_docs:
    all_tokens.extend(doc)

In [ ]:
from collections import Counter

s_vocab_counter = Counter(all_tokens)

In [ ]:
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}
for token, freq in s_vocab_counter.most_common():
    vocab[token] = len(vocab)

In [ ]:
def encode(tokens):
    return [
        vocab.get(token, vocab['<UNK>'])
        for token in tokens
    ]

In [ ]:
encoded_docs = tokenized_docs.apply(encode)

In [ ]:
print(s_vocab_counter.most_common(10))

[('▁', 17916), ('▁영화', 17908), ('▁너무', 8315), ('▁정말', 8056), ('▁진짜', 6273), ('▁이', 5111), ('▁왜', 3388), ('▁그냥', 3341), ('▁이런', 3289), ('▁더', 3274)]


### HuggingFace Tokenizer를 사용하여 단어사전 구축

In [ ]:
from tokenizers.pre_tokenizers import Whitespace
import pandas as pd

whitespace = Whitespace()

def bpe_tokenize(text):
    if pd.isnull(text):
        return []
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)

    tokens = whitespace.pre_tokenize_str(str(text))

    return [token for token, _ in tokens]

In [ ]:
h_start = time.perf_counter()
tokenized_docs = X.apply(bpe_tokenize)
h_end = time.perf_counter()

In [ ]:
from collections import Counter

all_tokens = []

for doc in tokenized_docs:
    all_tokens.extend(doc)

h_vocab_counter = Counter(all_tokens)

In [ ]:
print(h_vocab_counter.most_common(10))

[('영화', 17908), ('너무', 8315), ('정말', 8056), ('진짜', 6273), ('이', 5111), ('왜', 3388), ('그냥', 3341), ('이런', 3289), ('더', 3274), ('수', 2947)]


## 3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석

### 상위 30개 기준, 단어사전 조회하기

In [ ]:
# [[YOUR CODE]]
print("s_token\n")
top30 = s_vocab_counter.most_common(30)
for token, freq in top30:
    print(f"{token}")
print("h_token\n")
top30 = h_vocab_counter.most_common(30)
for token, freq in top30:
    print(f"{token}")

s_token

▁
▁영화
▁너무
▁정말
▁진짜
▁이
▁왜
▁그냥
▁이런
▁더
▁수
▁영화를
▁잘
▁다
▁보고
▁좀
▁영화는
▁영화가
▁그
▁본
▁최고의
▁봤는데
▁없는
▁내가
▁이건
▁없다
▁드라마
▁이렇게
▁완전
▁평점
h_token

영화
너무
정말
진짜
이
왜
그냥
이런
더
수
영화를
잘
다
보고
좀
영화는
영화가
그
본
최고의
봤는데
없는
내가
이건
없다
드라마
이렇게
완전
평점
이거


### 분절된 토큰 빈도 조회

In [ ]:
# [[YOUR CODE]]
print("s_frequency\n")
top30 = s_vocab_counter.most_common(30)
for token, freq in top30:
    print(f"{freq}")

print("h_frequency\n")
top30 = h_vocab_counter.most_common(30)
for token, freq in top30:
    print(f"{freq}")

s_frequency

17916
17908
8315
8056
6273
5111
3388
3341
3289
3274
2947
2800
2661
2650
2608
2559
2466
2454
2440
2307
2232
2125
2027
2026
1976
1888
1877
1835
1821
1798
h_frequency

17908
8315
8056
6273
5111
3388
3341
3289
3274
2947
2800
2661
2650
2608
2559
2466
2454
2440
2307
2232
2125
2027
2026
1976
1888
1877
1835
1821
1798
1779


## 4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정

### 단일 문장 변환 속도 측정

### 코퍼스 전체 변환 속도 측정

In [ ]:
# [[YOUR CODE]]
print(f"s-전체 처리 시간: {s_end-s_start:.4f} sec")
print(f"h-전체 처리 시간: {h_end-h_start:.4f} sec")

s-전체 처리 시간: 7.7548 sec
h-전체 처리 시간: 3.1505 sec


## 5. 기타 추가 실험

In [ ]:
# [[YOUR CODE]]

## 6. 사용성과 장단점 분석

**장점**  
음, 사실 결과에서 유의미한 차이는 못찾겠다. 다만 h의 경우 거의 두배의 속도가 측정되어 더 유리하게 느껴졌다. 공백을 _로 처리하는 이유에 대해서 더 잘 생각해봐야겠다.

**단점**